# Project Starter (Learner): Launch Strategy — Feature, Price, and Marketing Decisions from Customer Data

This is the **starter notebook** for the end-of-course project in *Customer & Behavioral Analytics*.

Your goal is to build a defensible launch plan for a new subscription tier by completing the tasks below.
You will write the analysis code yourself. This notebook provides structure, checkpoints, and hints.


## Scenario
You are a **Customer Analytics Lead** supporting a product team launching a **new subscription tier** for a consumer app.
Leadership needs a single, defensible launch plan:

- **Which features to bundle**
- **What to charge**
- **Which offer to run**
- **How to position the tier**
- **Where to spend marketing dollars**

You will analyze multiple datasets and synthesize a **stakeholder-ready memo**.

---

## What you will submit
Your submission should include:
1) This notebook (`.ipynb`) with outputs saved
2) An **Insight Memo**. You can write it in Markdown at the end of this notebook or as a separate `report.md`.

There is **no minimum length requirement**—use the provided memo template and ensure you answer each recommendation section clearly.

---

## Project Tasks (high level)
Complete the tasks below and connect them into a single recommendation.

### Task 1 — Preferences + Pricing 
Create a **choice-based preference model** with `choice_conjoint_tasks.csv`.

You must produce:
- feature trade-off insights (what drives selection)
- a bundle comparison using `candidate_bundles.csv`
- a price sensitivity chart with labeled axes
- a **directional WTP estimate** using the coefficient ratio approach (e.g., `-beta_feature / beta_price`) the numerical value must be reported

### Task 2 — Offer effectiveness (propensity/IPW)
Using `observational_offer_adoption.csv`, estimate incremental effect of the offer on:
- upgrade rate
- revenue

You must show:
- a propensity model using **at least 4 covariates total**, including **≥2 categorical** (choose from: `region`, `device`) and **≥2 numeric** (choose from: `age`, `tenure_days`, `prior_spend_12m`, `sessions_30d`)
- a propensity overlap plot (treated vs control) **and** a short markdown note commenting on whether overlap looks adequate
- balance diagnostics before vs after IPW (SMD table and/or plot for numeric covariates) **and** a short markdown note commenting on whether balance improved
- IPW ATE reported for **both outcomes**: upgrade rate and revenue

### Task 3 — Choose ONE: Positioning (MDS) OR Value Drivers (Factor Analysis)
Choose **one** of the following:

- **Option A: Positioning (MDS)** by deriving distances from `candidate_bundles.csv` (standardize features → Euclidean distances → create dissimilarity matrix **D**)
- **Option B: Value Drivers (Factor Analysis)** using `survey_value_drivers.csv` (fit **2 or 3 factors**)

You must translate the result into a positioning/messaging recommendation, including:
- **MDS path:** identify **(1) closest substitute pair** and **(1) whitespace opportunity**
- **Factor Analysis path:** list the **top 3 items per factor** (by absolute loading) and write **2–4 sentences** interpreting each factor

### Task 4 — Marketing budget (MMM)
Using `marketing_weekly_channels.csv`, build a simple MMM:
- time-aware split (train/test by time)
- carryover/adstock features for all 5 channels (spend_search, spend_social, spend_video, spend_affiliate, spend_email)
- time/seasonality controls
- Provide a directional channel summary (e.g., coefficient table or ranked channel list). Include the sentence:
    **“MMM results are correlational (directional), not causal.”**


### Task 5 — Final Insight Memo
Synthesize everything into a launch recommendation:
- bundle + price
- offer recommendation
- positioning/messaging (from Task 3)
- budget recommendation (MMM)
- assumptions + limitations + next steps

---


In [1]:
# -----------------------------
# Imports and configuration
# -----------------------------
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, mean_absolute_error
from sklearn.manifold import MDS
from sklearn.decomposition import FactorAnalysis

import statsmodels.api as sm

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

# ------------------------------------------------------------
# Project paths (preferred: your Google Drive project folder)
# ------------------------------------------------------------

CHOICE_PATH   = Path("choice_conjoint_tasks.csv")
BUNDLES_PATH  = Path("candidate_bundles.csv")
ADOPT_PATH    = Path("observational_offer_adoption.csv")
SURVEY_PATH   = Path("survey_value_drivers.csv")
MMM_PATH      = Path("marketing_weekly_channels.csv")
SIM_PATH      = Path("bundle_similarity_matrix.csv")


In [2]:
# -----------------------------
# Verify required files exist
# -----------------------------
required = [CHOICE_PATH, BUNDLES_PATH, ADOPT_PATH, MMM_PATH]
missing = [p for p in required if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Missing required project file(s):\n"
        + "\n".join([f" - {p}" for p in missing])
        + "\n\nPlace the CSVs in the Project_Data folder and rerun."
    )

print("All required files found.")
for p in required:
    print(" -", p.name)


All required files found.
 - choice_conjoint_tasks.csv
 - candidate_bundles.csv
 - observational_offer_adoption.csv
 - marketing_weekly_channels.csv


## Load datasets

In [ ]:
# TODO: Load the core datasets
df_choice   = pd.read_csv(CHOICE_PATH)
df_bundles  = pd.read_csv(BUNDLES_PATH)
df_adopt    = pd.read_csv(ADOPT_PATH)
df_mmm      = pd.read_csv(MMM_PATH, parse_dates=["week_start"])

# Optional datasets (only needed if you pick the corresponding option)
df_sim = pd.read_csv(SIM_PATH, index_col=0) if SIM_PATH.exists() else None
df_survey = pd.read_csv(SURVEY_PATH) if SURVEY_PATH.exists() else None

# Quick sanity checks (recommended)
display(df_choice.head())
display(df_bundles)
display(df_adopt.head())
display(df_mmm.head())


# Task 1 — Preferences + Pricing 

## Choice-based model (practical)
**Dataset:** `choice_conjoint_tasks.csv`

**Hint:** Each `(respondent_id, task_id)` contains multiple alternatives and exactly one has `chosen = 1`.

Suggested steps:
1) Sanity check each task has exactly one chosen alternative
2) Engineer usable features (e.g., `log(storage_gb)`, one-hot flags)
3) Fit a choice-based model that can score alternatives (logistic is acceptable for this project)
4) List and Interpret the top 3 drivers (coefficient signs + relative magnitude)
5) Create a price sensitivity curve and feature-lift chart with labeled axes
6) Score candidate bundles in `candidate_bundles.csv`
7) Compute directional WTP via `- beta_feature / beta_price` (report the numerical value of WTP for at least 1 feature, e.g., NoAds)


### Task 2 — Offer effectiveness (propensity + IPW)

**Dataset:** `observational_offer_adoption.csv`

Goal: Estimate incremental impact of receiving an offer on:
- `upgraded_to_new_tier`
- `new_tier_monthly_revenue`

Required evidence:
- propensity score model
- overlap plot (treated vs control)
- balance diagnostics (SMD before/after IPW for numeric covariates)
- adjusted effect estimate (IPW ATE on upgrade and revenue)

Covariates (required):
- Use **at least 4 covariates total**, including:
  - **at least 2 numeric** (choose from: `age`, `tenure_days`, `prior_spend_12m`, `sessions_30d`)
  - **at least 2 categorical** (choose from: `region`, `device`) encoded appropriately (e.g., one-hot)

You must define the following variables:
- Treatment: `offer_received`
- Outcomes: `upgraded_to_new_tier` and `new_tier_monthly_revenue`


# Task 3 — Choose ONE: Positioning (MDS) OR Value Drivers (Factor Analysis)

Pick **one** option below and complete it.

You will use the result to write a short recommendation on how to position the tier or what themes to emphasize in messaging.


## Option A: Positioning (MDS)
Build your dissimilarity matrix **D** using the following approach:
- **Approach:** Use `candidate_bundles.csv`, engineer numeric features (e.g., price, log(storage), binary flags), **standardize** columns, then compute **Euclidean distances** to form dissimilarity matrix **D**

Deliverables (required):
- Dissimilarity matrix **D** created using **Approach the above approach** (show a snippet or describe how it was built)
- 2D perceptual map (MDS plot) with labeled bundles
- Identify **(1) closest substitute pair** (nearest bundles)
- Identify **(1) whitespace opportunity** (a positioning direction that’s relatively distinct)


## Option B: Value Drivers (Factor Analysis)
Use `survey_value_drivers.csv` (Likert items).

Deliverables (required):
- Fit **FactorAnalysis with 2 or 3 factors**
- Provide a factor loadings table (items × factors)
- For each factor, list the **top 3 items by absolute loading**
- Provide a short interpretation paragraph per factor that translates into messaging/value-driver guidance

**Note:** Use `sklearn.decomposition.FactorAnalysis` for stability.


# Task 4 — Marketing budget (MMM)

**Dataset:** `marketing_weekly_channels.csv`

Goal: Provide directional channel insights and a cautious budget recommendation.

Required evidence:
- time-aware train/test split
- adstock/carryover features for **all** five channels (spend_search, spend_social, spend_video, spend_affiliate, spend_email)
- time/seasonality controls
- **Ridge regression model fit** (required)
- model fit check: **report test MAE AND include an Actual vs Predicted plot**
- Provide a directional channel summary (e.g., coefficient table or ranked channel list). Including a required caveat sentence:
  **“MMM results are correlational (directional), not causal.”**

Suggested channels:
- `spend_search`, `spend_social`, `spend_video`, `spend_affiliate`, `spend_email`
Target:
- `weekly_revenue_usd`


# Task 5 — Final Insight Memo

Write your memo here (or in a separate `report.md`). It must be decision-ready and defensible.

Include:
- **Bundle recommendation:** which bundle (or refined bundle) and why (evidence from Task 1)
- **Pricing recommendation:** what to charge + WTP logic (directional) from Task 1
- **Offer recommendation:** whether to scale/refine the offer based on IPW effects and diagnostics (Task 2)
- **Positioning / messaging:** based on Task 3 (MDS OR Factor Analysis)
- **Budget recommendation:** based on MMM (Task 4), with volatility caveat
- **Assumptions / limitations / next steps**

Use clear section headings and complete sentences.


## Memo Template (copy/paste and fill)

### Executive Summary
- (3–6 bullets) What we recommend and why

### Evidence and Findings
#### Preferences + Pricing (Task 1)
- Key drivers:
- Bundle comparison:
- Directional WTP:

#### Offer Impact (Task 2)
- Diagnostics (overlap + balance):
- IPW effect estimates (upgrade + revenue):

#### Positioning/Messaging (Task 3 — choose one)
- If MDS:
- If Factor Analysis:

#### Marketing Budget (Task 4)
- Model fit + caveats:
- Directional channel insights:
- Budget recommendation:

### Recommendations
1) Bundle:
2) Price:
3) Offer:
4) Positioning/Messaging:
5) Budget:

### Assumptions & Limitations
- (bullet list)

### Next Steps
- (bullet list)
